# Week 3, day 1 (afternoon) — Worksheet 07 SOLUTIONS: filtering   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) against the real
`data/sales.csv`, and the quoted output is what it actually printed.

Question 8 is the one to re-read. Two filters that read identically in English
return different numbers of rows.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Filtering. Run this once.
import pandas as pd

sales = pd.read_csv("data/sales.csv")

print("shape:", sales.shape)
print("columns:", list(sales.columns))
print()
print(sales[["OrderID", "Region", "Category", "Quantity", "Sales"]].head())

PART A — one condition

### Question 1

`sales["Sales"] > 1000` -> **59** orders of 300.

The mask is a 300-element boolean Series, one entry per row, and `.sum()`
counts the `True`s. Handing that mask back to the frame keeps those rows.

Getting the count before looking at the rows is a good habit: 59 of 300 is
a fifth of the file, so you know before you read anything that this is a
substantial subset rather than a handful of outliers.

In [ ]:
mask = sales["Sales"] > 1000
print("orders over 1000:", mask.sum())
print()
print(sales[mask][["OrderID", "Region", "Sales"]].head())

### Question 2

Ontario `80`, Prarie `75`, West `70`, Atlantic `33`, Quebec `18`, Northwest Territories `10`, Yukon `8`, Nunavut `6`. -> the filter returns `80`, matching.

The counts total 300, so every row has a region and nothing is missing.

Note the second entry: **`Prarie`**, not `Prairie`. That is a spelling
mistake in the source system, and it is now a fact about your data. If you
filter for `"Prairie"` you get zero rows and no error — the query is
perfectly valid, the value simply does not exist.

This is why `value_counts()` belongs in your first look at any categorical
column. It shows you the values that are actually present, spelled the way
they are actually spelled, instead of the ones you assume are there.

In [ ]:
counts = sales["Region"].value_counts()
print(counts)
print()
ontario = sales[sales["Region"] == "Ontario"]
print("filtered rows:", len(ontario))
print("value_counts said:", counts["Ontario"])

### Question 3

West's first surviving labels -> `[11, 12, 13, 14, 15, 16, 21, 22]`. `west.iloc[0]` -> OrderID **59815**; `sales.iloc[0]` -> **8710**.

The labels survived the filter unchanged — they start at 11 and skip 17-20,
because those rows were not West. Positions were renumbered from zero;
labels were not.

The Ontario line at the end is the warning. This file is grouped by
region and the Ontario block sits at the top, so `ontario.iloc[0]` returns
the same order as `sales.iloc[0]` — the bug is invisible for that one
filter and visible for every other. A test written against the Ontario
subset would pass and prove nothing.

After filtering, use `.loc` and labels, or call `reset_index(drop=True)`
to deliberately make positions meaningful again.

In [ ]:
west = sales[sales["Region"] == "West"]
print("first 8 surviving labels:", list(west.index[:8]))
print()
print(west.head(3)[["OrderID", "Region", "Sales"]])
print()
print("west.iloc[0]  OrderID:", west.iloc[0]["OrderID"])
print("sales.iloc[0] OrderID:", sales.iloc[0]["OrderID"])
print()
# The file is grouped by region, so the Ontario rows ARE the first rows --
# and that filter would have made position 0 agree by accident.
ontario = sales[sales["Region"] == "Ontario"]
print("ontario.iloc[0] OrderID:", ontario.iloc[0]["OrderID"], "<- agrees, by luck")

PART B — more than one condition

### Question 4

Ontario **and** over 1000 -> **13** rows.

13 out of the 80 Ontario orders, and out of the 59 large ones. An `and`
is always no larger than either half.

The parentheses are compulsory. `&` has higher precedence than `>` in
Python, so `sales["Sales"] > 1000 & sales["Region"] == "Ontario"` parses
as `sales["Sales"] > (1000 & sales["Region"]) == "Ontario"` — which is
nonsense and raises something unhelpful. Wrap each condition and the
problem never arises.

In [ ]:
mask = (sales["Region"] == "Ontario") & (sales["Sales"] > 1000)
print("Ontario AND over 1000:", mask.sum())
print()
print(sales[mask][["OrderID", "Region", "Sales"]].head())

### Question 5

Technology `68`, over-3000 `18`, naive sum **`86`**, actual OR **`76`**, counted twice **`10`**.

The two counts do not add up to the `or` count, and the 10-row gap is
exactly the rows that are both — Technology orders that are also over
3000. Adding the two counts counts those rows twice.

This is inclusion–exclusion, and it is the single most common way a
summary table ends up with a total larger than its population. Any time
you see category counts that sum to more than the number of records, the
categories overlap and someone added them.

The fix is not arithmetic, it is to compute the union directly with `|`
and let Pandas count each row once.

In [ ]:
tech = sales["Category"] == "Technology"
big = sales["Sales"] > 3000
either = tech | big

print("Technology:      ", tech.sum())
print("over 3000:       ", big.sum())
print("naive sum:       ", tech.sum() + big.sum())
print("actual OR count: ", either.sum())
print("counted twice:   ", (tech & big).sum())

### Question 6

`~(== "Ontario")` -> `220`. `!= "Ontario"` -> `220`. -> `220 + 80 = 300`.

Both spellings agree and they partition the file exactly. `~` inverts a
mask; `!=` builds the inverted mask directly. Prefer `!=` when you are
negating a single comparison, and `~` when you need to invert something
compound like `~((a) & (b))`.

The reason this partition is clean is that `Region` has no missing values
— Q2 showed the counts totalling 300. On a column with `NaN`, neither
`== x` nor `!= x` is true for the missing rows, so the two counts would
**not** add up to the total. That is worth checking before trusting a
complement.

In [ ]:
a = (~(sales["Region"] == "Ontario")).sum()
b = (sales["Region"] != "Ontario").sum()
print("with ~ :", a)
print("with !=:", b)
print("agree:  ", a == b)
print("and they total:", a + (sales["Region"] == "Ontario").sum(), "of", len(sales))

### Question 7

`.isin(["Ontario", "West", "Quebec"])` -> `168`, matching the three `|` conditions exactly.

Same answer, and `isin` scales. Three values written as `|` conditions is
already hard to scan; twenty is unreadable and easy to get wrong by one
missing pair of parentheses.

`isin` also takes the values from a variable, so the list can come from
another query, a config file, or another column — which the `|` form
cannot do without building the expression dynamically.

In [ ]:
wanted = ["Ontario", "West", "Quebec"]
by_isin = sales["Region"].isin(wanted).sum()

by_or = (
    (sales["Region"] == "Ontario")
    | (sales["Region"] == "West")
    | (sales["Region"] == "Quebec")
).sum()

print("isin:", by_isin)
print("or:  ", by_or)
print("agree:", by_isin == by_or)

### Question 8

`between(10,20)` -> **60**. Manual `>=`/`<=` -> **60**. `inclusive="neither"` -> **53**. -> **7** rows sit exactly on 10 or 20.

`between` is inclusive on both ends by default, which is why it matches
the manual version exactly.

The third number is the point. Seven of these orders have a quantity of
precisely 10 or precisely 20, and whether they belong in 'between 10 and
20' is a question the English sentence does not answer. Both 60 and 53 are
honest readings of the request.

That is a 12% swing in the answer, decided by a default nobody stated.
When a specification says 'between', it has not finished specifying — and
the person who wrote it usually has not noticed.

In [ ]:
b_default = sales["Quantity"].between(10, 20).sum()
b_manual = ((sales["Quantity"] >= 10) & (sales["Quantity"] <= 20)).sum()
b_neither = sales["Quantity"].between(10, 20, inclusive="neither").sum()

print("between(10,20)                 :", b_default)
print("(>=10) & (<=20)                :", b_manual)
print("between(10,20,'neither')       :", b_neither)
print()
print("default matches the manual one:", b_default == b_manual)
print("rows sitting exactly on 10 or 20:", b_default - b_neither)

### Question 9

13 rows, total `28250.7785`, mean `2173.1368076923077`.

All three numbers are correct and the mean is the dangerous one.

The subset was defined as 'orders over 1000', so its mean **cannot** be
below 1000 — not because of anything about Ontario, but because of how the
rows were chosen. Reporting `2173` as 'the average Ontario order' would be
arithmetically impeccable and substantively false; the real average Ontario
order includes the 67 rows you filtered out.

This is the shape of most misleading statistics: a correct calculation on
a self-selected population. The defence is to always report the
denominator next to the number. '13 orders' makes the reader ask 'out of
how many?' — and `2173` on its own does not.

In [ ]:
mask = (sales["Region"] == "Ontario") & (sales["Sales"] > 1000)
subset = sales[mask]

print("rows: ", len(subset))
print("total:", subset["Sales"].sum())
print("mean: ", subset["Sales"].mean())

# The mean is a real number computed from a real subset. It is also the
# average of a self-selected group -- orders ALREADY known to exceed 1000 --
# so it cannot be smaller than 1000 no matter what the data says. Quoting it
# as "the average Ontario order" would be arithmetically correct and
# substantively false.

### Question 10

Using `and` -> **raises** `ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().`

Python's `and` needs each side to be a single `True` or `False` so it can
decide which one to return. You gave it a 300-element Series, and there is
no defensible way to collapse that to one boolean — some rows are Ontario
and some are not — so Pandas refuses rather than guessing.

The suggestions in the message are a red herring for this situation.
`.any()` and `.all()` would each produce a single boolean and make the
error go away, and both would give you a completely wrong filter. What you
actually want is `&`, which combines the masks element by element and is
not mentioned in the message at all.

Same for `or` -> `|`, and `not` -> `~`. This is one of the few places
where reading the exception and doing what it says leads you further from
the answer.

In [ ]:
print(sales[(sales["Region"] == "Ontario") and (sales["Sales"] > 1000)])